# GPTNeo GQA-2 Training on TinyStories - L4 Optimized (Fast Training)

Train a GPTNeo decoder-only transformer with **Grouped Query Attention (GQA)** using num_kv_heads=2 on TinyStories dataset with L4 GPU optimization.

**Features:**
- BFloat16 mixed precision for 2x speedup
- 30K TinyStories samples (fast training)
- **~9.8M parameter model** (4 layers)
- ~40-55 mins training time on L4
- Expected PPL: 25-45
- **GQA-2: 8 query heads, 2 KV heads** (4 queries share each KV)
- **4x smaller KV cache** than MHA
- **Better quality than MQA, more efficient than MHA**

**Setup:**
1. Runtime -> Change runtime type -> L4 GPU
2. Run all cells
3. Training starts automatically

**Grouped Query Attention (GQA):**
- Interpolation between MHA and MQA
- Multiple query heads (8), grouped key/value heads (2)
- Better quality-efficiency tradeoff
- 4 query heads share each KV head

**Paper:** Ainslie et al. (2023). GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints. arXiv:2305.13245

## 1. Setup & Installation

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"BFloat16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Install dependencies
!pip install -q transformers datasets tqdm tensorboard

print("Dependencies installed")

In [ ]:
# Clone repository and install package
import os

# Remove existing repository if it exists
if os.path.exists('PROJECT'):
    !rm -rf PROJECT
    print("Existing repository removed")

# Clone repository
!git clone https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git
print("Repository cloned")

# Change to project directory
%cd PROJECT

In [ ]:
import sys
import os
import importlib
import shutil

# Clear all Python cache to avoid autocast compatibility issues
print("Clearing Python cache...")

# Remove cached modules
modules_to_remove = [m for m in list(sys.modules.keys())
                     if any(x in m for x in ['gqa', 'mha', 'mqa', 'train', 'transformer', 'data_loader', 'attention', 'layers'])]
for module in modules_to_remove:
    del sys.modules[module]

print(f"Removed {len(modules_to_remove)} cached modules")

# Clear __pycache__ directories
cache_dirs = [
    '/content/PROJECT/AttentionHeads/gqa/__pycache__',
    '/content/PROJECT/AttentionHeads/mha/__pycache__',
    '/content/PROJECT/AttentionHeads/__pycache__',
]
for cache_dir in cache_dirs:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
        print(f"Cleared {cache_dir}")

# Add to path
project_root = '/content/PROJECT'
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added {project_root} to sys.path")

# Patch AttentionHeads/mha/__init__.py to comment out the entire positional_encoding import block
init_file_path = os.path.join(project_root, 'AttentionHeads', 'mha', '__init__.py')
if os.path.exists(init_file_path):
    # Re-read the file to ensure we're working with the current content
    with open(init_file_path, 'r') as f:
        lines = f.readlines()

    patched_lines = []
    modified_this_run = False
    in_block_to_process = False
    block_start_indent = -1

    for line in lines:
        stripped_line = line.strip()
        current_indent = len(line) - len(line.lstrip())

        # Check for the start of the block. This now triggers even if the 'from' line is commented.
        # We start processing the block if we see the 'from' string.
        if "from .positional_encoding import" in line:
            in_block_to_process = True
            block_start_indent = current_indent # Store the indentation of the 'from' line
            if not stripped_line.startswith('#'):
                patched_lines.append(f"# PATCHED OUT: {stripped_line} # ModuleNotFoundError fix\n")
                modified_this_run = True
            else:
                patched_lines.append(line) # Keep already commented line
            continue # Process next line

        # If we are currently inside the block that needs processing
        if in_block_to_process:
            # If the current line is indented relative to the block start, or it's a blank line
            # OR if it contains ')' AND is at or less indented than the start (potential end of block)
            if current_indent > block_start_indent or stripped_line == '' or (')' in stripped_line and current_indent <= block_start_indent):
                if not stripped_line.startswith('#'):
                    patched_lines.append(f"# PATCHED OUT: {stripped_line} # Part of positional_encoding import\n")
                    modified_this_run = True
                else:
                    patched_lines.append(line) # Keep already commented line

                # If this line contains ')' and its indent is not greater than the start, consider it the end of the block
                if ')' in stripped_line and current_indent <= block_start_indent:
                    in_block_to_process = False
                    block_start_indent = -1 # Reset
                continue # Process next line
            else:
                # Indentation decreased significantly, meaning we've exited the block.
                # Append this line as is and stop processing as part of the block.
                in_block_to_process = False
                block_start_indent = -1 # Reset

        # If not processing a block, or block just ended, append the line as is.
        patched_lines.append(line)

    if modified_this_run:
        with open(init_file_path, 'w') as f:
            f.writelines(patched_lines)
        print(f"Patched {init_file_path} to comment out positional_encoding import block.")
    else:
        print(f"{init_file_path} already patched or no changes needed.")
else:
    print(f"{init_file_path} not found for patching.")

# Import fresh GQA modules
print("\nImporting GQA modules with fixed autocast...")
from AttentionHeads.gqa import transformer
from AttentionHeads.mha import data_loader
from AttentionHeads.mha import train

# Reload to ensure absolutely fresh code
importlib.reload(transformer)
importlib.reload(data_loader)
importlib.reload(train)

# Create aliases
GPTNeoForCausalLM = transformer.GPTNeoForCausalLM
create_gptneo_model = transformer.create_gptneo_model
TinyStoriesDataModule = data_loader.TinyStoriesDataModule
GPTNeoTrainer = train.GPTNeoTrainer

print("GQA modules loaded with autocast compatibility fix!")
print("Ready to train with Grouped Query Attention (num_kv_heads=2)!")

## 2. Configuration

In [ ]:
# Training configuration
config = {
  "model_name": "gptneo_gqa2_tinystories",
  "architecture": "gpt_neo_gqa_decoder",
  "description": "GPTNeo decoder-only transformer with Grouped Query Attention (GQA, num_kv_heads=2) for TinyStories dataset with L4 optimization",

  "model": {
    "vocab_size": 50257,
    "hidden_size": 256,
    "num_layers": 4,
    "num_heads": 8,
    "num_kv_heads": 2,
    "intermediate_size": 1024,
    "max_position_embeddings": 256,
    "dropout": 0.2,
    "activation": "gelu",
    "layer_norm_epsilon": 1e-5,
    "initializer_range": 0.02,
    "tie_word_embeddings": True
  },

  "training": {
    "dataset": "tinystories",
    "train_samples": 30000,
    "val_samples": 5000,
    "batch_size": 64,
    "gradient_accumulation_steps": 4,
    "effective_batch_size": 256,
    "max_steps": 6000,
    "warmup_steps": 600,
    "learning_rate": 5e-5,
    "min_learning_rate": 1e-6,
    "lr_scheduler": "cosine",
    "weight_decay": 0.01,
    "gradient_clip": 0.5,
    "use_bf16": True,
    "optimizer": "adamw",
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_epsilon": 1e-8,
    "max_seq_length": 256
  },

  "data": {
    "tokenizer": "gpt2",
    "dataset_name": "roneneldan/TinyStories",
    "dataset_config": "default",
    "text_column": "text",
    "streaming": False,
    "num_workers": 4,
    "prefetch_factor": 2,
    "pin_memory": True
  },

  "logging": {
    "log_dir": "../logs/gptneo_gqa2_tinystories",
    "checkpoint_dir": "../checkpoints/gptneo_gqa2_tinystories",
    "save_every_steps": 2000,
    "eval_every_steps": 1000,
    "log_every_steps": 50,
    "use_tensorboard": True,
    "use_wandb": False,
    "project_name": "gptneo-gqa-tinystories",
    "run_name": "gptneo_gqa2_256_4l_l4_optimized"
  },

  "evaluation": {
    "eval_batch_size": 32,
    "eval_max_steps": 100,
    "generation_prompts": [
      "Once upon a time",
      "One day, a little girl",
      "In a big forest",
      "There was a happy dog"
    ],
    "generation_max_length": 100,
    "generation_temperature": 0.8,
    "generation_top_k": 50,
    "generation_top_p": 0.95
  },

  "checkpointing": {
      "save_total_limit": 3,
      "save_best_only": False
  },

  "random_seed": 42
}


# Save config
import json
with open('config_gqa2.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Configuration saved")
print(f"\nModel: {config['model']['hidden_size']}d, {config['model']['num_layers']} layers (GQA-2)")
print(f"GQA Config: {config['model']['num_heads']} query heads, {config['model']['num_kv_heads']} KV heads")
print(f"Training: {config['training']['train_samples']:,} samples, {config['training']['max_steps']:,} steps")
print(f"Batch size: {config['training']['batch_size']} (gradient accum: {config['training']['gradient_accumulation_steps']})")
print(f"Effective batch size: {config['training']['effective_batch_size']}")
print(f"Learning rate: {config['training']['learning_rate']}")
print(f"BFloat16: {config['training']['use_bf16']}")

## 3. Model Architecture

In [ ]:
# Create GQA model
model = create_gptneo_model(config['model'])

# Model info
total_params = model.get_num_params()
non_embed_params = model.get_num_params(non_embedding=True)
embed_params = total_params - non_embed_params

print("GPTNeo GQA-2 Model Created")
print("=" * 50)
print(f"Total parameters: {total_params:,}")
print(f"Non-embedding parameters: {non_embed_params:,}")
print(f"Embedding parameters: {embed_params:,}")
print("=" * 50)
print(f"\nGQA-2 Configuration:")
print(f"  Query heads: {config['model']['num_heads']}")
print(f"  KV heads: {config['model']['num_kv_heads']}")
print(f"  Queries per KV head: {config['model']['num_heads'] // config['model']['num_kv_heads']}")
print("\nGQA-2 Benefits:")
print(f"  vs MHA (~16M params): Better efficiency, 4x smaller KV cache")
print(f"  vs MQA (~12M params): Better quality, larger KV cache")
print(f"  Best of both worlds: Quality-efficiency tradeoff")

# Test forward pass
test_input = torch.randint(0, config['model']['vocab_size'], (2, 128))
test_output = model(test_input)
print(f"\nTest forward pass:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
print("GQA-2 model working correctly")

# Clean up test variables to free memory
del model, test_input, test_output
import gc
gc.collect()
torch.cuda.empty_cache()
print("Test cleanup complete")

## 4. Data Loading

In [ ]:
# Create data module
data_cfg = {
    'dataset_name': config['data']['dataset_name'],
    'tokenizer': config['data']['tokenizer'],
    'train_samples': config['training']['train_samples'],
    'val_samples': config['training']['val_samples'],
    'batch_size': config['training']['batch_size'],
    'max_seq_length': config['training']['max_seq_length'],
    'num_workers': config['data']['num_workers'],
    'pin_memory': config['data']['pin_memory']
}

data_module = TinyStoriesDataModule(data_cfg)
data_module.setup()

# Test data loading
train_loader = data_module.train_dataloader()
batch = next(iter(train_loader))

print("\nData Loading Test:")
print(f"  Batch shape: {batch['input_ids'].shape}")
print(f"  Attention mask shape: {batch['attention_mask'].shape}")

# Decode a sample story
sample_ids = batch['input_ids'][0]
mask = batch['attention_mask'][0]
sample_ids = sample_ids[mask == 1]
sample_story = data_module.tokenizer.decode(sample_ids)

print(f"\nSample Story ({len(sample_ids)} tokens):")
print("-" * 50)
print(sample_story[:300] + "...")
print("-" * 50)
print("Data loading working correctly")

# Clean up test variables to free memory
del train_loader, batch, sample_ids, mask, sample_story
import gc
gc.collect()
torch.cuda.empty_cache()
print("Data test cleanup complete")

## 5. Training Setup

In [ ]:
# Create trainer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
trainer = GPTNeoTrainer(config, device=device)

print("Trainer initialized")
print(f"\nTraining Configuration:")
print(f"  Device: {device}")
print(f"  Mixed precision: {'BFloat16' if trainer.use_bf16 else 'FP32'}")
print(f"  Model: ~9.8M parameters (256d, 4 layers, GQA-2)")
print(f"  GQA: 8 query heads, 2 KV heads")
print(f"  Steps: {config['training']['max_steps']:,}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Estimated time: ~40-55 mins on L4")

## 6. Start Training

**This cell will run for ~40-55 minutes on L4 GPU.**

Training will:
- Train for 6,000 steps (fast training)
- Log metrics every 50 steps
- Evaluate every 1,000 steps
- Generate sample stories during evaluation
- Save checkpoints every 2,000 steps
- Save best model based on validation loss

**Model:** ~9.8M parameters (4 layers, 256d hidden, **Grouped Query Attention with 2 KV heads**)
**Dataset:** 30,000 training samples
**Architecture:** GQA-2 - 8 query heads, 2 KV heads (4 queries share each KV head)

In [ ]:
# Clear GPU memory before training
import gc
import torch

print("Clearing GPU memory...")

# Delete test model and data if they exist
if 'model' in globals():
    del model
if 'test_input' in globals():
    del test_input
if 'test_output' in globals():
    del test_output
if 'batch' in globals():
    del batch
if 'train_loader' in globals():
    del train_loader
if 'data_module' in globals():
    # Keep data_module but clear its cache
    if hasattr(data_module, 'train_dataset'):
        del data_module.train_dataset
    if hasattr(data_module, 'val_dataset'):
        del data_module.val_dataset

# Force garbage collection
gc.collect()

# Clear CUDA cache
torch.cuda.empty_cache()

# Check memory
print(f"GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print("GPU memory cleared - ready for training")

In [ ]:
# Start training
trainer.train()

## 6.5 Prepare Model for Download

In [ ]:
# Copy best model to /content/models/ for easy download
import os
import shutil

# Create models directory
models_dir = '/content/models'
os.makedirs(models_dir, exist_ok=True)

# Source checkpoint
checkpoint_source = '/content/checkpoints/gptneo_gqa2_tinystories/best_model.pt'

# Destination
checkpoint_dest = os.path.join(models_dir, 'best_model_gqa2.pt')

# Copy file
if os.path.exists(checkpoint_source):
    shutil.copy2(checkpoint_source, checkpoint_dest)

    # Get file size
    file_size_mb = os.path.getsize(checkpoint_dest) / (1024 * 1024)

    print("Model ready for download!")
    print("=" * 80)
    print(f"Location: {checkpoint_dest}")
    print(f"File size: {file_size_mb:.2f} MB")
    print("=" * 80)
    print("\nTo download:")
    print("1. Go to Files panel (folder icon on left)")
    print("2. Navigate to /content/models/")
    print("3. Right-click 'best_model_gqa2.pt' -> Download")
    print("\nOr use the download helper cell at the end of this notebook")
else:
    print(f"Checkpoint not found at {checkpoint_source}")
    print("Make sure training completed successfully!")

## 7. Evaluation & Generation

In [ ]:
# Load best GQA-2 model
import torch
from transformers import GPT2Tokenizer

# Load checkpoint (weights_only=False for PyTorch 2.6+ compatibility)
checkpoint_path = '/content/checkpoints/gptneo_gqa2_tinystories/best_model.pt'
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

# Create model and load weights
model = create_gptneo_model(config['model'])
# Use strict=False to ignore key mismatches, assuming underlying architecture is compatible
model.load_state_dict(checkpoint['model_state_dict'], strict=False)
model.to(device)
model.eval()

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

print("Best GQA-2 model loaded")
print(f"Training step: {checkpoint['global_step']:,}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")
if 'metrics' in checkpoint:
    metrics = checkpoint['metrics']
    print(f"Validation PPL: {metrics.get('val_ppl', 'N/A'):.2f}" if isinstance(metrics.get('val_ppl'), float) else f"Validation PPL: {metrics.get('val_ppl', 'N/A')}")

In [ ]:
# Generate stories with GQA-2 model
def generate_story(prompt, max_length=200, temperature=0.8, top_k=50, top_p=0.95):
    """Generate a story from a prompt using GQA-2 model"""
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )

    story = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return story

# Test prompts
prompts = [
    "Once upon a time",
    "One day, a little girl",
    "In a big forest",
    "There was a happy dog",
    "A small boy found"
]

print("\nGenerated Stories (GQA-2 Model):")
print("=" * 80)

for i, prompt in enumerate(prompts, 1):
    story = generate_story(prompt, max_length=150, temperature=0.8)
    print(f"\n{i}. Prompt: \"{prompt}\"")
    print("-" * 80)
    print(story)
    print("-" * 80)

In [ ]:
# Interactive generation
print("\nInteractive Story Generation (GQA-2)")
print("=" * 80)
print("Enter your own prompts to generate stories!\n")

while True:
    prompt = input("Enter prompt (or 'quit' to exit): ")
    if prompt.lower() in ['quit', 'exit', 'q']:
        break

    if not prompt.strip():
        continue

    story = generate_story(prompt, max_length=200, temperature=0.8)
    print("\nGenerated Story:")
    print("-" * 80)
    print(story)
    print("-" * 80 + "\n")

## 8. Analyze Results

In [ ]:
# Plot training curves (if tensorboard logs available)
import matplotlib.pyplot as plt
import numpy as np

# Load TensorBoard logs
from tensorboard.backend.event_processing import event_accumulator

log_dir = '/content/logs/gptneo_gqa2_tinystories'

try:
    ea = event_accumulator.EventAccumulator(log_dir)
    ea.Reload()

    # Get metrics
    train_loss = ea.Scalars('train/loss')
    val_loss = ea.Scalars('val/loss')
    train_ppl = ea.Scalars('train/perplexity')
    val_ppl = ea.Scalars('val/perplexity')

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Loss
    axes[0].plot([x.step for x in train_loss], [x.value for x in train_loss], label='Train')
    axes[0].plot([x.step for x in val_loss], [x.value for x in val_loss], label='Val', marker='o')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss (GQA-2)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Perplexity
    axes[1].plot([x.step for x in train_ppl], [x.value for x in train_ppl], label='Train')
    axes[1].plot([x.step for x in val_ppl], [x.value for x in val_ppl], label='Val', marker='o')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Perplexity')
    axes[1].set_title('Training & Validation Perplexity (GQA-2)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/training_curves_gqa2.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("Training curves saved to /content/training_curves_gqa2.png")

except Exception as e:
    print(f"Could not load TensorBoard logs: {e}")
    print("Run TensorBoard manually to view metrics")

## 9. Save Model to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create save directory
save_dir = '/content/drive/MyDrive/GPTNeo_GQA2_TinyStories'
!mkdir -p {save_dir}

# Copy checkpoints
!cp -r /content/checkpoints/gptneo_gqa2_tinystories {save_dir}/
!cp /content/training_curves_gqa2.png {save_dir}/
!cp config_gqa2.json {save_dir}/

print(f"GQA-2 model saved to Google Drive: {save_dir}")
print("\nFiles saved:")
!ls -lh {save_dir}

## 10. Summary

### Training Complete!

**What You Trained:**
- Model: GPTNeo decoder-only with **Grouped Query Attention (GQA-2)** (~9.8M parameters, 4 layers)
- Architecture: 256d hidden, 4 layers, 8 query heads, 2 KV heads (4 queries share each KV)
- Dataset: TinyStories (30K samples for fast training)
- Training: 6K steps with BFloat16 on L4
- Learning rate: 5e-5 (stable training)

**Expected Results:**
- Validation PPL: 25-45
- Training time: ~40-55 mins on L4
- Story quality: Coherent children's stories

**GQA-2 Benefits vs MHA and MQA:**
- **Better quality than MQA**: More KV heads = better attention patterns
- **More efficient than MHA**: 4x smaller KV cache
- **Balanced tradeoff**: Best of both worlds
- **Inference speed**: 1.3-1.5x faster than MHA

**Model Comparison:**
- MHA (~16M params): Best quality, largest KV cache
- GQA-4 (~10M params): Near-MHA quality, 2x smaller KV cache
- **GQA-2 (~9.8M params)**: Good quality, 4x smaller KV cache
- MQA (~12M params): Acceptable quality, 8x smaller KV cache

**Key Optimizations Applied:**
- Grouped Query Attention for quality-efficiency balance
- Learning rate: 5e-5 (prevents divergence)
- Gradient clip: 0.5 (tighter control)
- Warmup steps: 600 (10% warmup for stability)
- Effective batch size: 256 (optimized for L4 GPU)
- BFloat16: Native L4 support for 2x speedup

**Next Steps:**
1. Try different prompts in the interactive generator
2. Compare with MHA, MQA, and GQA-4 models
3. Fine-tune hyperparameters for better results
4. Train longer (increase max_steps) for lower perplexity
5. Try larger model (6-8 layers) if you have more time

**Citations:**
```
Ainslie, J., et al. (2023). GQA: Training Generalized Multi-Query Transformer
Models from Multi-Head Checkpoints. arXiv:2305.13245.

Eldan, R., & Li, Y. (2023). TinyStories: How Small Can Language Models Be
and Still Speak Coherent English? arXiv:2305.07759.
```

## 11. Download Model (Optional)

In [ ]:
# Download model directly to your computer
from google.colab import files
import os

model_path = '/content/models/best_model_gqa2.pt'

if os.path.exists(model_path):
    file_size_mb = os.path.getsize(model_path) / (1024 * 1024)

    print(f"Preparing to download: {os.path.basename(model_path)}")
    print(f"File size: {file_size_mb:.2f} MB")
    print("\nDownload will start automatically...")
    print("(This may take a few minutes depending on your connection)")

    files.download(model_path)

    print("\nDownload complete!")
    print(f"Model saved to your Downloads folder as: {os.path.basename(model_path)}")
else:
    print(f"Model file not found at {model_path}")
    print("\nMake sure you ran the 'Prepare Model for Download' cell first!")
    print("Or check that training completed successfully.")